<a href="https://colab.research.google.com/github/guitorte/audio/blob/ccr-8be874fb-rdbsnf/neural-upscaler/Neural_Audio_Upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎚️ Neural Audio Upscaler

Repairs songs from your Google Drive **without re-generating them**. It's the same recording, with the damage from MP3/AAC compression and missing high end repaired.

```
song.mp3 ─► ① BS-RoFormer ─┬─ vocals ────────► ② Apollo ─┐
                           └─ instrumental* ─► ② Apollo ─┴─► ③ AudioSR ─► ④ Matchering ─► song_upscaled.wav
                              * = mix − vocals, so nothing is lost by splitting
```

| Step | What it fixes | Default |
|---|---|---|
| ① **Vocal split** (BS-RoFormer) | lets each part be restored on its own and lets you rebalance the vocal | on |
| ② **Apollo** | compression artefacts: swirly cymbals, smeared transients, "underwater" highs | on |
| ③ **AudioSR** | the missing top end above the MP3 shelf (e.g. everything over 16 kHz), output at 48 kHz | off (slow) |
| ④ **Matchering** | mastering: copies the loudness, EQ, and width of a reference song you choose | off |

AudioSR only **adds** frequencies above the detected cutoff. Everything below it stays the original audio. Files that are already full band skip it automatically.

**How to use:** `Runtime ▸ Change runtime type ▸ T4 GPU` (L4 is faster), then run the 3 cells in order.

In [ ]:
#@title 1 · Connect Google Drive and list your songs
DRIVE_FOLDER = "áudio"  #@param {type:"string"}
#@markdown Path inside *My Drive* (e.g. `áudio` or `Music/áudio`). Accents and capitals don't need to match exactly.

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

REPO, BRANCH = "/content/audio", "ccr-8be874fb-rdbsnf"
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                       "https://github.com/guitorte/audio.git", REPO]).returncode:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/guitorte/audio.git", REPO], check=True)
sys.path.insert(0, f"{REPO}/neural-upscaler")
import upscaler as U

FOLDER = U.find_folder(DRIVE_FOLDER)
FILES = U.list_audio(FOLDER)
print(f"📁 {FOLDER}  ({len(FILES)} songs)\n")
for i, f in enumerate(FILES, 1):
    print(f"{i:3d}. {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError:
    gpu = None
print(f"\nGPU: {gpu or '⚠️ none. Use Runtime ▸ Change runtime type ▸ T4 GPU'}")

In [ ]:
#@title 2 · Choose songs and steps, then run
SONGS = "all"  #@param {type:"string"}
#@markdown `all`, numbers from the list (`1, 3, 5-7`), or parts of names (`know better; demo`).
SEPARATE_VOCALS = True  #@param {type:"boolean"}
APOLLO = True  #@param {type:"boolean"}
AUDIOSR = False  #@param {type:"boolean"}
#@markdown ⏱ AudioSR is slow: expect 10+ minutes per song on a T4 (faster on an L4). The first run also spends a few minutes setting up its own environment.
AUDIOSR_STEPS = 50  #@param {type:"slider", min:20, max:200, step:10}
CUTOFF_HZ = 0  #@param {type:"integer"}
#@markdown `0` = detect the lossy shelf automatically. Set e.g. `16000` to force AudioSR to rebuild everything above it.
VOCAL_GAIN_DB = 0.0  #@param {type:"slider", min:-6, max:6, step:0.5}
MASTER_REFERENCE = ""  #@param {type:"string"}
#@markdown Optional: number or name of a song **in the same folder** whose sound you want to match. Empty = no mastering.
KEEP_PARTS = False  #@param {type:"boolean"}
#@markdown Also save the restored vocal and instrumental stems.
OVERWRITE = False  #@param {type:"boolean"}

selected = U.select_files(FILES, SONGS)
print("Processing:", *[f"  • {f.name}" for f in selected], sep="\n")
U.setup_main(SEPARATE_VOCALS, APOLLO, bool(MASTER_REFERENCE.strip()))
opts = U.Options(separate=SEPARATE_VOCALS, apollo=APOLLO, audiosr=AUDIOSR, audiosr_steps=AUDIOSR_STEPS,
                 cutoff_hz=float(CUTOFF_HZ), vocal_gain_db=VOCAL_GAIN_DB,
                 reference=MASTER_REFERENCE, keep_parts=KEEP_PARTS, overwrite=OVERWRITE)
RESULTS = U.process(selected, FOLDER, opts)

print("\n" + "═" * 60)
for r in RESULTS:
    print(f"{os.path.basename(r.source)}\n   → {r.output}")
    for n in r.notes + ([r.skipped] if r.skipped else []):
        print(f"     · {n}")

In [ ]:
#@title 3 · Compare before / after (level-matched, so louder doesn't fool your ears)
START_AT_SECONDS = 45  #@param {type:"number"}
CLIP_SECONDS = 30  #@param {type:"number"}

import numpy as np, matplotlib.pyplot as plt, subprocess
from IPython.display import Audio, Markdown, display

def excerpt(path, sr):
    raw = subprocess.run(["ffmpeg", "-v", "error", "-ss", str(START_AT_SECONDS), "-t", str(CLIP_SECONDS),
                          "-i", str(path), "-ac", "2", "-ar", str(sr), "-f", "f32le", "-"],
                         capture_output=True, check=True).stdout
    return np.frombuffer(raw, dtype=np.float32).reshape(-1, 2)

for r in RESULTS:
    if not r.output:
        continue
    sr = U.sf.info(r.output).samplerate
    before, after = excerpt(r.source, sr), excerpt(r.output, sr)
    after = after * (U.rms(before) / max(U.rms(after), 1e-9))
    display(Markdown(f"### {os.path.basename(r.source)}"))
    display(Markdown("**Before**")); display(Audio(np.clip(before.T, -1, 1), rate=sr, normalize=False))
    display(Markdown("**After** (level-matched)")); display(Audio(np.clip(after.T, -1, 1), rate=sr, normalize=False))
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.6), sharey=True)
    for ax, audio, title in ((axes[0], before, "Before"), (axes[1], after, "After")):
        ax.specgram(audio.mean(axis=1), NFFT=2048, Fs=sr, noverlap=1024, cmap="magma", vmin=-130, vmax=-20)
        ax.set_title(title); ax.set_xlabel("seconds"); ax.set_ylim(0, sr / 2)
    axes[0].set_ylabel("Hz"); plt.tight_layout(); plt.show()

### Tips
- **Compare first, then switch on more steps.** Start with Apollo only, listen, then try AudioSR on the song that sounds dullest.
- The spectrogram shows the fix: an MP3 has a hard horizontal "ceiling" around 16 kHz, and AudioSR fills the space above it.
- Results are saved as 24-bit WAV in `upscaled/` inside your folder, together with a `.json` of the settings used.
- AudioSR *invents* the new highs. If they sound hissy or metallic, the Apollo-only version is the safer choice.
- A song that is already full band (e.g. a lossless master) skips AudioSR. Apollo and mastering still apply.